In [0]:
%python
# dml/05_carga_stg_scoring_fof.ipynb
# %%
from datetime import datetime

catalogo = "product_dev"
schema = "financas"

print("Iniciando cálculo de Scoring e Ranking REAL para FIIs de Fundos de Fundos (FoF)...")

# %%
# 1. Busca e calcula as métricas reais cruzando Yahoo Finance com Staging CVM
qry_calculo_metricas = f"""
  WITH historico_recente AS (
    -- Avalia a liquidez recente dos últimos 30 dias
    SELECT 
      ticker,
      preco_fechamento,
      volume_negociado,
      data_pregao
    FROM {catalogo}.{schema}.stg_historico_cotacoes_fiis
    WHERE data_pregao >= ADD_MONTHS(CURRENT_DATE(), -1)
  ),
  
  liquidez_fiis AS (
    -- Filtra apenas fundos com liquidez real ativa
    SELECT 
      ticker,
      AVG(volume_negociado) AS volume_medio_diario,
      MAX(data_pregao) AS data_ultimo_negocio
    FROM historico_recente
    WHERE volume_negociado > 0
    GROUP BY ticker
    HAVING volume_medio_diario >= 50 AND data_ultimo_negocio >= DATE_SUB(CURRENT_DATE(), 15)
  ),

  historico_12m AS (
    -- Coleta cotações e dividendos pagos dos últimos 12 meses
    SELECT 
      ticker,
      preco_fechamento,
      proventos_pagos,
      data_pregao
    FROM {catalogo}.{schema}.stg_historico_cotacoes_fiis
    WHERE data_pregao >= ADD_MONTHS(CURRENT_DATE(), -12)
  ),
  
  precos_atuais AS (
    -- Captura o último preço de mercado ativo
    SELECT ticker, preco_fechamento AS preco_atual
    FROM (
      SELECT ticker, preco_fechamento, ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY data_pregao DESC) as rn
      FROM {catalogo}.{schema}.stg_historico_cotacoes_fiis
      WHERE volume_negociado > 0
    ) WHERE rn = 1
  ),
  
  dividendos_12m AS (
    -- Soma dos dividendos de mercado dos últimos 12 meses
    SELECT ticker, SUM(proventos_pagos) AS total_dividendos_12m
    FROM historico_12m
    GROUP BY ticker
  ),
  
  cadastro_fof AS (
    -- Filtra apenas FIIs de FoF ativos
    SELECT ticker, nome_fundo, classificacao
    FROM {catalogo}.{schema}.dim_fundo_imobiliario
    WHERE classificacao = 'Fundo de Fundos (FoF)'
  )
  
  -- Junta mercado (Yahoo) com contabilidade oficial (CVM)
  SELECT 
    c.ticker,
    p.preco_atual,
    comp.valor_patrimonial_cota,
    comp.patrimonio_liquido,
    comp.percentual_taxa_administracao,
    ap.valor_investido_outros_fiis,
    COALESCE(d.total_dividendos_12m, 0.0) AS total_dividendos_12m
  FROM cadastro_fof c
  INNER JOIN liquidez_fiis l ON c.ticker = l.ticker
  INNER JOIN precos_atuais p ON c.ticker = p.ticker
  LEFT JOIN dividendos_12m d ON c.ticker = d.ticker
  -- INNER JOIN com as novas Stgings CVM
  INNER JOIN {catalogo}.{schema}.stg_cvm_informe_complemento comp ON c.ticker = comp.ticker
  INNER JOIN {catalogo}.{schema}.stg_cvm_informe_ativo_passivo ap ON c.ticker = ap.ticker
"""

df_metricas = spark.sql(qry_calculo_metricas)
df_metricas.createOrReplaceTempView("v_metricas_base_fof_real")

# %%
# 2. Aplicação das Regras de Scoring e Peso Real para FoF
qry_scoring = f"""
  WITH limites AS (
    SELECT 
      MAX(total_dividendos_12m) as max_div,
      MIN(total_dividendos_12m) as min_div,
      MAX(valor_investido_outros_fiis) as max_alocacao,
      MIN(valor_investido_outros_fiis) as min_alocacao,
      MAX(percentual_taxa_administracao) as max_taxa,
      MIN(percentual_taxa_administracao) as min_taxa
    FROM v_metricas_base_fof_real
  ),
  
  scores_calculados AS (
    SELECT 
      m.ticker,
      m.preco_atual,
      m.valor_patrimonial_cota,
      ROUND(m.preco_atual / m.valor_patrimonial_cota, 2) AS p_vp,
      ROUND((m.total_dividendos_12m / m.preco_atual) * 100.0, 2) AS dividend_yield_12m,
      m.valor_investido_outros_fiis,
      m.percentual_taxa_administracao,
      
      -- Normalização do Dividend Yield (Maior = Melhor)
      ROUND(COALESCE(((m.total_dividendos_12m - l.min_div) / NULLIF(l.max_div - l.min_div, 0)) * 100.0, 0.0), 2) AS nota_dy,
      
      -- Normalização da Alocação em outros FIIs (Maior = Melhor/Foco real do FoF)
      ROUND(COALESCE(((m.valor_investido_outros_fiis - l.min_alocacao) / NULLIF(l.max_alocacao - l.min_alocacao, 0)) * 100.0, 0.0), 2) AS nota_alocacao,
      
      -- Normalização da Taxa de Administração (Menor Taxa = Melhor Nota para evitar bitributação)
      ROUND(COALESCE(((l.max_taxa - m.percentual_taxa_administracao) / NULLIF(l.max_taxa - l.min_taxa, 0)) * 100.0, 0.0), 2) AS nota_taxa,
      
      -- Normalização para P/VP em FoF (Se P/VP estiver entre 0.82 e 0.95, ganha nota 100)
      CASE 
        WHEN (m.preco_atual / m.valor_patrimonial_cota) BETWEEN 0.82 AND 0.95 THEN 100.0
        WHEN (m.preco_atual / m.valor_patrimonial_cota) < 0.82 
          THEN ROUND(GREATEST(0.0, (1.0 - (0.82 - (m.preco_atual / m.valor_patrimonial_cota)) * 3.0) * 100.0), 2)
        ELSE ROUND(GREATEST(0.0, (1.0 - ((m.preco_atual / m.valor_patrimonial_cota) - 0.95) * 5.0) * 100.0), 2)
      END AS nota_pvp
    FROM v_metricas_base_fof_real m
    CROSS JOIN limites l
  )
  
  SELECT 
    ticker,
    CURRENT_DATE() AS data_referencia,
    preco_atual,
    valor_patrimonial_cota,
    p_vp,
    dividend_yield_12m,
    valor_investido_outros_fiis,
    percentual_taxa_administracao,
    -- Média Ponderada Real: 40% P/VP, 30% DY, 20% Alocação em FIIs (Duplo Desconto Real), 10% Menor Taxa Adm
    ROUND((nota_pvp * 0.40) + (nota_dy * 0.30) + (nota_alocacao * 0.20) + (nota_taxa * 0.10), 2) AS score_final
  FROM scores_calculados
"""

df_scores = spark.sql(qry_scoring)
df_scores.createOrReplaceTempView("v_scores_fof_reais_calculados")

# %%
# 3. Geração do Ranking Geral e carga física com INSERT OVERWRITE
qry_insert_ranking_fof = f"""
  INSERT OVERWRITE {catalogo}.{schema}.stg_scoring_fof
  SELECT 
    ticker,
    data_referencia,
    preco_atual,
    valor_patrimonial_cota,
    p_vp,
    dividend_yield_12m,
    valor_investido_outros_fiis AS duplo_desconto_estimado, -- Mapeia a alocação real de FIIs
    percentual_taxa_administracao AS taxa_administracao_ano, -- Taxa contábil real CVM
    score_final,
    ROW_NUMBER() OVER (ORDER BY score_final DESC) AS posicao_ranking,
    CURRENT_TIMESTAMP() AS data_calculo
  FROM v_scores_fof_reais_calculados
"""

print(f"Gravando classificação e ranking REAL de FoF em: {catalogo}.{schema}.stg_scoring_fof...")
spark.sql(qry_insert_ranking_fof)
print("✅ Cálculo de Scoring REAL e Ranking de FoF finalizado com SUCESSO!")